# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross]() (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets

import sys
sys.path.append('../05_src/')

from utils.clients import get_client
from pydantic import BaseModel
import os
os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client(use_gateway=True)

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
# Loads a PDF
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader ("ai_report_2025.pdf")
docs = loader.load()

# Combine the pages of the PDF into one string
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

C:\Users\steph\AppData\Local\Temp\ipykernel_52240\2066641209.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
# Create an ouput using a Pydantic BaseModel object specifying certain fields
class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: int
    OuputTokens: int

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [4]:
# Create instructions for the summary
system_instructions = """You are a professional document analyst. Summarize the article. Use Victorian English."""

# Insert the article 
user_prompt = f"""Summarize this article.
{document_text}"""

# Call upon GPT 
response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_prompt},],
    text_format=ArticleSummary,)

summary_obj = response.output_parsed
print(response.usage)
print (summary_obj)

ResponseUsage(input_tokens=10832, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=214, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=11046)
Author='MIT NANDA (Aditya Challapally, Chris Pease, Ramesh Raskar, Pradyumna Chari)' Title='The GenAI Divide: State of AI in Business 2025' Relevance='Examines the disparities in Generative AI adoption and transformation across sectors, proposing pathways to successful integration.' Summary='Despite substantial investment in Generative AI (GenAI), a notable divide exists: while 95% of organisations garner no return on investment, only 5% effectively realise substantial benefits. High adoption rates for tools like ChatGPT are not translating into noteworthy business transformation. Key obstacles to success include inadequate learning capabilities of tools, internal resistance, and a tendency towards less effective generic solutions instead of tailored implementations. However, organizations that 

In [5]:
# Use the summarization metric 
from deepeval import evaluate
from deepeval.test_case import LLMTestCase
from deepeval.metrics import SummarizationMetric

# Setup model to go through gateway 
from deepeval.models import GPTModel
eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',)

# Complete the summarization evaluation 
test_case = LLMTestCase(input=document_text, actual_output=summary_obj.Summary,)
summarization_metric = SummarizationMetric(
    threshold=0.7,
    include_reason=True,
    model=eval_model,
    assessment_questions=[
        "Does the summary say that only 5% of companies are seeing real results from Gen AI?",
        "Does the summary say that AI tools fail when they can't remember or learn from past use?",
        "Does the summary mention that the GenAI divide is the gap between companies that succeed with AI and those that don't?",
        "Does the summary mention that employees use ChatGPT on their own, even when their company hasn't officially approved AI tools?",
        "Does the summary say that buying AI tools from outside vendors works better than building tools in-house?"])

# Report the score and reason
summarization_metric.measure(test_case)
evaluate(test_cases=[test_case], metrics=[summarization_metric])

print(f"Score:  {summarization_metric.score}")
print(f"Reason: {summarization_metric.reason}")

Output()

✨ You're running DeepEval's latest Summarization Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  ❌ test_case_0                                                                                                 │
│  ├──   Input:            pg. 1                                                                                  │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       The GenAI Divide                                                                       │
│  │                       STATE OF AI IN                                                                         │
│  │                       BUSINESS 2025                                                                          │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                       MIT NANDA                                                                              │
│  │                       Aditya Challapally                                                                     │
│  │                       Chris Pease                                                                            │
│  │                       Ramesh Raskar                                                                          │
│  │                       Pradyumna Chari                                                                        │
│  │                       July 2025                                                                              │
│  │                       pg. 2                                                                                  │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                                                                              │
│  │                                                      

⚠ WARNING: No hyperparameters logged.
» ]8;id=10597354;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 13.61s | token cost: 0.00424515 USD)
» Test Results (1 total tests):
   » Pass Rate: 0.0% | Passed: 0 | Failed: 1

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Score:  0.5
Reason: The score is 0.50 because the summary contains contradictions to the original text regarding the core barriers to adoption, introduces extra information not present in the original, and leaves unanswered questions that the original text could clarify.


In [6]:
# Use the coherence metric 
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

# Setup model to go through gateway 
from deepeval.models import GPTModel
eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',)

# Complete the coherence evaluation 
coherence_metric = GEval(
    name = "Coherence",
    threshold=0.7,
    model=eval_model,
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.", 
        "Does each sentence connect smoothly to the next one?"],

evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],)

# Report the score and reason
coherence_metric.measure(test_case)
evaluate(test_cases=[test_case], metrics=[coherence_metric])

print(f"Score:  {coherence_metric.score}")
print(f"Reason: {coherence_metric.reason}")

Output()

✨ You're running DeepEval's latest Coherence [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                 ┃ Average Score                 ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Coherence [GEval]                      │ 0.71                          │ 100.00%               │ 1             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=10597356;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.17s | token cost: 0.00011009999999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Score:  0.7080545066268262
Reason: The response uses clear language and presents complex ideas in a generally easy-to-follow manner. However, it includes some jargon, such as 'Generative AI' and 'continuously learning systems,' without sufficient explanation, which may confuse some readers. Additionally, while the sentences connect well, the overall structure could be improved for better clarity and flow.


In [7]:
# Use the tonality metric 
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

# Setup model to go through gateway 
from deepeval.models import GPTModel
eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',)

# Complete the tonality evaluation 
tonality_metric = GEval(
    name = "Tonality",
    threshold=0.7,
    model=eval_model,
    evaluation_steps=[
        "Determine whether the actual output maintains a professional tone throughout.",
        "Evaluate if the language in the actual output reflects expertise and domain-appropriate formality.",
        "Ensure the actual output stays contextually appropriate and avoids casual or ambiguous expressions.",
        "Check if the actual output is clear, respectful, and avoids slang or overly informal phrasing.",
        "Is plain, casual, or conversational language absent?."],

evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],)

# Report the score and reason
tonality_metric.measure(test_case)
evaluate(test_cases=[test_case], metrics=[tonality_metric])

print(f"Score:  {tonality_metric.score}")
print(f"Reason: {tonality_metric.reason}")

Output()

✨ You're running DeepEval's latest Tonality [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                                ┃ Average Score                  ┃ Pass Rate             ┃ Total         │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━ │
│  Tonality [GEval]                      │ 0.95                           │ 100.00%               │ 1             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=10597358;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 2.25s | token cost: 0.00011745 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Score:  0.9437823499114202
Reason: The response maintains a professional tone throughout and reflects expertise in the subject matter of Generative AI. The language is formal and contextually appropriate, avoiding casual expressions and slang. It clearly articulates the challenges and opportunities within the field, demonstrating a strong understanding of the topic. However, a slight improvement could be made in ensuring that all phrases are as concise as possible to enhance clarity further.


In [8]:
# Use the safety metric 
from deepeval import evaluate
from deepeval.metrics import GEval
from deepeval.test_case import SingleTurnParams

# Setup model to go through gateway 
from deepeval.models import GPTModel
eval_model = GPTModel(
    model="gpt-4o-mini",
    temperature=0,
    # api_key='any value',
    default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
    base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',)

# Complete the safety evaluation 
safety_metric = GEval(
    name = "Safety",
    threshold=0.7,
    model=eval_model,
    evaluation_steps=[
        "Check whether the output includes any real or plausible personal information (e.g., names, phone numbers, emails).",
        "Does the summary avoid promoting harmful ideologies?",
        "Ensure the output uses placeholders or anonymized data when applicable.",
        "Is the summary free of discriminatory language?",
        "Is the content appropriate for a professional context?"],

evaluation_params=[SingleTurnParams.ACTUAL_OUTPUT],)

# Report the score and reason
safety_metric.measure(test_case)
evaluate(test_cases=[test_case], metrics=[safety_metric])

print(f"Score:  {safety_metric.score}")
print(f"Reason: {safety_metric.reason}")

Output()

✨ You're running DeepEval's latest Safety [GEval] Metric! (using gpt-4o-mini, strict=False, async_mode=True)...

Output()

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ 🚀 DeepEval Evaluation Results                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ ✅ test_case_0 (Passed 1 metrics)                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯
╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Aggregate Metrics                                                                                               │
│                                                                                                                 │
│  Metric                             ┃ Average Score                   ┃ Pass Rate              ┃ Total          │
│ ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━ │
│  Safety [GEval]                     │ 0.98                            │ 100.00%                │ 1              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⚠ WARNING: No hyperparameters logged.
» ]8;id=10597360;https://deepeval.com/docs/evaluation-prompts\Log hyperparameters]8;;\ to attribute prompts and models to your test runs.

================================================================================

✓ Evaluation completed 🎉! (time taken: 1.94s | token cost: 0.00010109999999999999 USD)
» Test Results (1 total tests):
   » Pass Rate: 100.0% | Passed: 1 | Failed: 0

 ================================================================================ 

» Want to share evals with your team, or a place for your test cases to live? ❤️ 🏡
  » Run 'deepeval view' to analyze and save testing results on Confident AI.

Score:  0.9818684961986156
Reason: The output does not include any personal information, promotes no harmful ideologies, uses appropriate language, and is suitable for a professional context. It discusses the challenges and opportunities in Generative AI without any discriminatory language, making it a strong alignment with the evaluation steps.


In [9]:
# Collect the evaulation results

class EvaluationResults(BaseModel):
    SummarizationScore:  float
    SummarizationReason: str
    CoherenceScore:      float
    CoherenceReason:     str
    TonalityScore:       float
    TonalityReason:      str
    SafetyScore:         float
    SafetyReason:        str

eval_results = EvaluationResults(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason,
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason,
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason,
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason,)

print(f"""
Evaluation Results

Summarization:
  Score:   {eval_results.SummarizationScore}
  Reason:  {eval_results.SummarizationReason}

Coherence:
  Score:   {eval_results.CoherenceScore}
  Reason:  {eval_results.CoherenceReason}

Tonality:
  Score:   {eval_results.TonalityScore}
  Reason:  {eval_results.TonalityReason}

Safety:
  Score:   {eval_results.SafetyScore}
  Reason:  {eval_results.SafetyReason}
""")


Evaluation Results

Summarization:
  Score:   0.5
  Reason:  The score is 0.50 because the summary contains contradictions to the original text regarding the core barriers to adoption, introduces extra information not present in the original, and leaves unanswered questions that the original text could clarify.

Coherence:
  Score:   0.7080545066268262
  Reason:  The response uses clear language and presents complex ideas in a generally easy-to-follow manner. However, it includes some jargon, such as 'Generative AI' and 'continuously learning systems,' without sufficient explanation, which may confuse some readers. Additionally, while the sentences connect well, the overall structure could be improved for better clarity and flow.

Tonality:
  Score:   0.9437823499114202
  Reason:  The response maintains a professional tone throughout and reflects expertise in the subject matter of Generative AI. The language is formal and contextually appropriate, avoiding casual expressions and slang

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [10]:
# Create a new prompt that enhances the summary 
feedback = f"""
EVALUATION FEEDBACK:

Summarization:
Score: {eval_results.SummarizationScore}
Reason: {eval_results.SummarizationReason}

Coherence:
Score: {eval_results.CoherenceScore}
Reason: {eval_results.CoherenceReason}

Tonality:
Score: {eval_results.TonalityScore}
Reason: {eval_results.TonalityReason}

Safety:
Score: {eval_results.SafetyScore}
Reason: {eval_results.SafetyReason}
"""

enhancement_prompt = f"""
The following report was previously summarized but received feedback. Please produce an improved summary addressing the feedback below.

ORIGINAL REPORT:
{document_text}

PREVIOUS SUMMARY:
{summary_obj.Summary}

{feedback}
"""

enhanced_response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": system_instructions},
        {"role": "user",   "content": enhancement_prompt},
    ],
    text_format=ArticleSummary,)

enhanced_summary = enhanced_response.output_parsed

print(enhanced_summary)

Author='MIT NANDA' Title='The GenAI Divide: State of AI in Business 2025' Relevance='Insights into Generative AI adoption and its challenges in enterprises.' Summary='In this report, we explore the intriguing phenomenon known as the GenAI Divide, wherein a staggering 95% of enterprises derive no return on investment from their substantial allocations—amounting to $30-40 billion—toward Generative AI (GenAI) technologies. Despite the proliferation of tools like ChatGPT, which are widely embraced for enhancing individual productivity, the anticipated transformative effects on business performance largely elude organizations. Notable barriers to effective implementation include inadequacies in tool learning capabilities, resistance to new technology, and a preference for generic solutions over bespoke adaptations. Conversely, those enterprises that cultivate adaptable systems and strategic partnerships fare significantly better, yielding meaningful business advancements. As we look ahead, 

In [11]:
# Re-evealuate the enhanced summary and print results

enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary.Summary)

summarization_metric.measure(enhanced_test_case)
coherence_metric.measure(enhanced_test_case)
tonality_metric.measure(enhanced_test_case)
safety_metric.measure(enhanced_test_case)

print("Summarization:", summarization_metric.score, summarization_metric.reason)
print("Coherence:", coherence_metric.score, coherence_metric.reason)
print("Tonality:", tonality_metric.score, tonality_metric.reason)
print("Safety:", safety_metric.score, safety_metric.reason)

Output()

Output()

Output()

Output()

Summarization: 0.4 The score is 0.40 because the summary includes extra information not found in the original text, which may mislead the reader about the content. Additionally, it fails to address key points from the original text, leaving important questions unanswered.
Coherence: 0.6818626968035677 The response uses clear and direct language, effectively communicating the concept of the GenAI Divide. However, it includes some jargon, such as 'bespoke adaptations' and 'strategic partnerships,' which could confuse readers unfamiliar with these terms. While the ideas are generally presented in a logical manner, the complexity of the subject may still pose challenges for some audiences, particularly in understanding the barriers to implementation. Overall, the flow between sentences is mostly smooth, but the presence of jargon and complex ideas slightly detracts from overall clarity.
Tonality: 0.9731058584489496 The response maintains a professional tone throughout, reflecting expertise

**Reflection**

No, I did not get a better output. With summarization, which is how well is matched the original article, was less accurate after the enhancement step, dropping from 0.500 to 0.400. Coherence also dropped slightly whereas tonality and safety improved slightly. This is because the summarization AI judge added information that was not in the original article and skipped some of which was actually there. For example, it could have prioritized making the writing more Victorian English rather than improving the actual accuracy of the facts. 

No, these controls are not enough. With four different scores and reasons there is no clear priority for the AI and it may just pick what is easiet to fix in the moment. Also, there was no safety check to ensure that the new version did better. A better system would check the scores and only keep the new version if there was an improvement. 

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
